# 05 · Backpressure and bounded queues

**Time:** about 25 minutes. **Before this:** lessons 01–04, especially threads.
**You need:** Python's standard library. No GPU needed.

You will see a backlog grow, run a small producer/consumer pipeline, and
estimate how much batch data can wait in memory. All examples are deliberately small.

[Course guide](README.md) · Previous: [Asyncio](04_asyncio.ipynb) ·
Next: [PyTorch DataLoader](06_pytorch_dataloader.ipynb)

## 1. A faster producer can create a problem

A **producer** creates work. A **consumer** uses it. A **queue** holds work
between them. `Queue` removes items in the order they were added: **first in,
first out (FIFO)**. With several consumers, completion order can still differ.

In training, CPU workers prepare batches and a GPU consumes them. A **batch** is a group of samples processed together.
**Host RAM** is the computer's main memory, where the CPU keeps its data.
A GPU has its own device memory in the setup we are describing.

```text
CPU workers → [ready batch] [ready batch] [ready batch] → GPU
                waiting data uses host RAM
```

If producers make 10 batches/second and the consumer handles 4, the backlog
grows by 6 batches/second while those rates continue. **Backpressure** means
slowing or pausing producers when downstream work cannot keep up.

**Predict:** after five seconds, how many extra batches will be waiting?

In [1]:
producer_rate = 10  # batches per second
consumer_rate = 4
batch_mib = 16  # Hypothetical batch payload, not allocated here.
backlog = 0

print("second | waiting batches | payload MiB")
for second in range(1, 6):
    backlog = max(0, backlog + producer_rate - consumer_rate)
    print(f"{second:>6} | {backlog:>15} | {backlog * batch_mib:>11}")

second | waiting batches | payload MiB
     1 |               6 |          96
     2 |              12 |         192
     3 |              18 |         288
     4 |              24 |         384
     5 |              30 |         480


There are 30 waiting batches: 480 MiB of payload in this simplified model.
We calculated counts only; we did not allocate that memory. **MiB** means
1,048,576 bytes. Python objects and other buffers would add more memory.

An unbounded queue has no configured item limit. It does not have infinite RAM.
A **bounded queue** has a limit. Once it fills, producers must wait for space.
The wait carries the consumer's speed back to the producer.

## 2. See the rule before starting threads

`queue.Queue` is safe to use between threads. Its `maxsize` limits the number
of queued items, not bytes. A size of zero means no configured limit.
`put()` adds an item and waits if full. `get()` removes an item and waits if empty.
The `nowait` variants raise an exception instead of waiting.
[Reference: Python queue](https://docs.python.org/3/library/queue.html).

In [2]:
from queue import Full, Queue

small_queue = Queue(maxsize=2)
small_queue.put("batch A")
small_queue.put("batch B")

try:
    small_queue.put_nowait("batch C")
except Full:
    print("Full: a normal put() would wait for a consumer.")

first = small_queue.get()
small_queue.task_done()
small_queue.put_nowait("batch C")
print("Removed:", first)
print("Now batch C fits.")

# Finish this tiny example so no unfinished items remain.
for _ in range(2):
    small_queue.get()
    small_queue.task_done()
small_queue.join()

Full: a normal put() would wait for a consumer.
Removed: batch A
Now batch C fits.


`task_done()` says that one removed item has been processed. `join()` waits
until all added items have matching `task_done()` calls. Getting an item and
finishing it are different events. `Queue.join()` waits for item
acknowledgments; `Thread.join()` from lesson 02 waits for a thread to finish.

## 3. Build a tiny training-pipeline stand-in

One thread will prepare 12 small byte buffers. Another will consume them.
Sleeps represent the durations of preparation and GPU work; **this does not
perform CPU parallel preprocessing or real GPU training**. Threads let us
isolate the queue behavior without needing hardware or data.

The functions are split into short cells so you can read each step.

In [3]:
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter, sleep

BATCH_BYTES = 64 * 1024  # Only 64 KiB per toy batch.
END = None  # An end marker (also called a sentinel): no more batches will arrive.

def produce(queue, count, prepare_seconds):
    put_seconds = 0.0
    observed_sizes = []
    for batch_id in range(count):
        sleep(prepare_seconds)
        batch = (batch_id, bytes(BATCH_BYTES))
        start = perf_counter()
        queue.put(batch, timeout=10)
        put_seconds += perf_counter() - start
        observed_sizes.append(queue.qsize())
        del batch  # Drop the producer's reference after sending.
    queue.put(END, timeout=10)
    return put_seconds, observed_sizes

`put_seconds` includes the time spent waiting for queue space plus small call
overheads. `qsize()` is only an approximate snapshot during concurrent work;
we use it for observation, never to decide whether a `put()` is safe.
A timeout makes a broken example fail instead of waiting forever.
`bytes(BATCH_BYTES)` creates a zero-filled buffer of that size. This buffer is
the **payload**, the actual batch data. A thread queue passes a reference to
the existing buffer. `del batch` removes the producer's reference after sending
it; any other references keep the buffer alive for as long as it is needed.

In [4]:
def consume(queue, consume_seconds):
    seen_ids = []
    while True:
        item = queue.get(timeout=10)
        try:
            if item is END:
                return seen_ids
            batch_id, payload = item
            assert len(payload) == BATCH_BYTES
            sleep(consume_seconds)
            seen_ids.append(batch_id)  # Keep small IDs, not all the buffers.
            del payload, item  # Release this consumer's buffer reference.
        finally:
            queue.task_done()  # Includes the end marker.

`finally` runs even if processing raises an error. We acknowledge every item
we take. `item is END` checks for our end marker; it is separate from batch data.
Acknowledging an item does not prove it was processed successfully. We read
the futures' results below so worker errors are visible.
This is a finite teaching pipeline; a service would also need a plan for
stopping its other workers after a failure, as covered in lessons 09 and 12.

In [5]:
def run_pipeline(capacity, count=12, prepare_seconds=0.01, consume_seconds=0.05):
    queue = Queue(maxsize=capacity)
    start = perf_counter()
    with ThreadPoolExecutor(max_workers=2) as pool:
        producer = pool.submit(produce, queue, count, prepare_seconds)
        consumer = pool.submit(consume, queue, consume_seconds)
        put_seconds, sizes = producer.result()
        seen_ids = consumer.result()
    queue.join()
    assert seen_ids == list(range(count))  # No missing, repeated, or reordered batches.
    return {
        "capacity": capacity,
        "seconds": perf_counter() - start,
        "put_seconds": put_seconds,
        "largest_snapshot": max(sizes, default=0),
        "completed": len(seen_ids),
    }

**Predict:** which run will spend more time inside `put()`? Will the bounded
version make the slow consumer faster?

Even the unbounded run below produces only 12 tiny buffers (768 KiB total payload).
Never demonstrate memory exhaustion by deliberately exhausting your machine.

In [6]:
unbounded = run_pipeline(capacity=0)
bounded = run_pipeline(capacity=2)

for name, result in [("unbounded", unbounded), ("bounded", bounded)]:
    print(f"{name:>9}: elapsed={result['seconds']:.3f}s, "
          f"time in put={result['put_seconds']:.3f}s, "
          f"largest queue snapshot={result['largest_snapshot']}, "
          f"finished={result['completed']}")

unbounded: elapsed=0.614s, time in put=0.000s, largest queue snapshot=9, finished=12
  bounded: elapsed=0.617s, time in put=0.339s, largest queue snapshot=2, finished=12


Usually, the bounded producer waits longer and the observed queue stays small.
Total time is similar because the consumer still needs about `12 * 0.05 = 0.6`
seconds of work. Scheduling changes exact numbers; we do not assert a timing result.

A bounded queue limits waiting work. It does not speed up the bottleneck.
A small buffer can still help: it keeps the consumer supplied through brief
changes in preparation time. If it is too small, the consumer may wait for data.

## 4. Queue capacity is not a total memory limit

The producer can hold a batch while waiting to put it in the queue. The
consumer can hold a batch after removing it. Both are outside the queue.

For our one-producer, one-consumer design with fixed-size batches, a rough
payload budget is `(queue capacity + 1 + 1) * batch bytes`. It leaves out Python
overhead, temporary copies, datasets, and library buffers. More workers can
hold more batches outside the queue. Variable batch sizes need extra care.

In [7]:
# float32 uses four bytes per value. This is a size calculation, not an allocation.
batch_size, channels, height, width = 32, 3, 224, 224
bytes_per_value = 4
batch_bytes = batch_size * channels * height * width * bytes_per_value
batch_mib = batch_bytes / 1024**2

print(f"One image batch payload: {batch_mib:.2f} MiB")
for capacity in [1, 2, 8]:
    waiting_mib = capacity * batch_mib
    pipeline_mib = (capacity + 2) * batch_mib
    print(f"Capacity {capacity}: queued payload <= {waiting_mib:.1f} MiB; "
          f"rough active + queued payload <= {pipeline_mib:.1f} MiB")

One image batch payload: 18.38 MiB
Capacity 1: queued payload <= 18.4 MiB; rough active + queued payload <= 55.1 MiB
Capacity 2: queued payload <= 36.8 MiB; rough active + queued payload <= 73.5 MiB
Capacity 8: queued payload <= 147.0 MiB; rough active + queued payload <= 183.8 MiB


## 5. Limit work before it piles up

A limit must apply at the place where work accumulates:

| Mechanism | What to limit |
| --- | --- |
| Threads | Use a bounded `queue.Queue` feeding a fixed number of workers |
| Processes | Limit pending submissions or use a bounded process queue; include data converted to bytes for transfer |
| Asyncio | Use a bounded `asyncio.Queue` feeding a fixed number of tasks |
| PyTorch DataLoader | Start with few workers and a small `prefetch_factor` (batches prepared ahead per worker); measure batch memory |

`max_workers=2` limits running workers, but repeatedly calling `submit()` can
still create millions of pending futures. A semaphore limits active operations,
but creating millions of asyncio tasks first still uses memory.

Python 3.14 adds `Executor.map(..., buffersize=...)` to limit submitted work whose
results have not yet been returned to the caller. Earlier versions need another design, such
as processing finite groups or explicitly keeping a small set of pending futures.
[Executor reference](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map).
[Asyncio queue reference](https://docs.python.org/3/library/asyncio-queue.html).

## Try it yourself

1. Run the pipeline with capacities 1, 2, and 8. Keep the work unchanged.
   Compare time inside `put()` and total time.
2. Make preparation slower than consumption. Which side should wait now?
3. Eight CPU workers produce 40 batches/second, but a GPU consumes 10.
   Does increasing the queue capacity fix long-term throughput? What should you measure?
4. A queue holds four batches of 20 MiB each. Is 80 MiB a bound on the program's RAM?

In [8]:
# Experiment here. Try prepare_seconds=0.06 and consume_seconds=0.01.
experiment = run_pipeline(capacity=1, count=8)
print(experiment)

{'capacity': 1, 'seconds': 0.4153204000322148, 'put_seconds': 0.2307339000981301, 'largest_snapshot': 1, 'completed': 8}


<details>
<summary>Worked answers — open after trying</summary>

1. A larger queue usually lets the producer finish sooner. The slow consumer
   still determines most of the total time.
2. Use `run_pipeline(2, count=8, prepare_seconds=0.06, consume_seconds=0.01)`.
   The consumer now waits for the producer; adding queue slots cannot create data sooner.
3. No. The GPU still completes at most 10 batches/second in this scenario. Measure
   batches completed per second, time waiting for input, and host/GPU memory.
   Cap pending work; investigate the slow stage before adding workers.
4. No. 80 MiB covers only four waiting payloads. Active batches, worker memory,
   dataset objects, temporary arrays, and the model also use memory. **Pinned
   buffers** are CPU memory kept in RAM, which can help GPU transfers;
   lesson 06 explains them.

</details>

## Before moving on

Explain why making producers wait can be the correct behavior. Then describe
the difference between limiting worker count, waiting item count, and bytes.

Next: [06 · PyTorch DataLoader](06_pytorch_dataloader.ipynb), where workers and
prefetching apply these ideas to batches for an ML model.